# LLM Judge

## Setup

In [ ]:
import os
import mlflow
from backend.constants import LLM_JUDGE, MONITORING_PATH

db_path = MONITORING_PATH / "mlflow.db"
if db_path.exists():
    os.chmod(db_path, 0o666)
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("rag_evaluation")

/Users/leolindqvistkrohnert/Fullstack_LLMops_Project-1/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/Users/leolindqvistkrohnert/Fullstack_LLMops_Project-1/src/study_buddy/monitoring/mlruns/2', creation_time=1778241919470, experiment_id='2', last_update_time=1778241919470, lifecycle_stage='active', name='rag_evaluation', tags={}, trace_location=None, workspace='default'>

## Dataset and predict_fn

In [3]:
import lancedb
import asyncio
import nest_asyncio
from backend.constants import VECTOR_DB_PATH
from backend.agents import bot_answer

nest_asyncio.apply()

vector_db = lancedb.connect(uri=VECTOR_DB_PATH)
docs = vector_db["LectureTranscript"].to_pandas()

evaluation_dataset = [
    {
        "inputs": {
            "prompt": row["document_name"],
            "context": row["content"],
        },
    }
    for _, row in docs.head(2).iterrows()
]

def predict_fn(prompt, context=None):
    result = asyncio.get_event_loop().run_until_complete(bot_answer(prompt))
    return result.answer

## Scorers and evaluation

In [ ]:
import requests
import json
from mlflow.genai import evaluate
from mlflow.genai.scorers import scorer

RELEVANCE_PROMPT = """Rate from 1-5 how well the answer addresses the question.
Question: {inputs}
Answer: {outputs}
Respond ONLY with JSON: {{"score": <1-5>}}"""

GROUNDEDNESS_PROMPT = """You are evaluating if an answer is grounded in the provided context.

STRICT RULES:
- Use ONLY the context below. Do NOT use any outside knowledge.
- Score 5: every claim in the answer is directly supported by the context
- Score 3: most claims are supported, but some are not in the context
- Score 1: the answer contains claims that are NOT in the context (hallucinations)

Question: {inputs}
Answer: {outputs}

Context (the only source of truth):
{context}

Respond ONLY with JSON: {{"score": <1-5>, "reason": "<brief explanation>"}}"""


def judge(prompt: str) -> int:
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        json={
            "model": "openai/gpt-4o-mini",
            "messages": [{"role": "user", "content": prompt}],
        },
    )
    text = response.json()["choices"][0]["message"]["content"]
    try:
        return int(json.loads(text)["score"])
    except Exception:
        return 0


@scorer
def relevance(inputs, outputs):
    return judge(RELEVANCE_PROMPT.format(inputs=inputs, outputs=outputs))


@scorer
def groundedness(inputs, outputs):
    return judge(GROUNDEDNESS_PROMPT.format(
        inputs=inputs["prompt"],
        outputs=outputs,
        context=inputs["context"]
    ))


scorers = [relevance, groundedness]

with mlflow.start_run(run_name="rag_evaluation"):
    results = evaluate(data=evaluation_dataset, predict_fn=predict_fn, scorers=scorers)

results

2026/05/08 18:15:18 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/05/08 18:15:18 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 2/2 [Elapsed: 00:09, Remaining: 00:00] [predict_fn: 45%, scorers: 55%]


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: rag_evaluation
  Run ID: 5e9e43ecae164941b311984898abd6f0

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



EvaluationResult(
  run_id: 5e9e43ecae164941b311984898abd6f0
  metrics:
    relevance/mean: 3.5
    groundedness/mean: 5.0
  result_df: 2 rows x 14 cols
)